In [0]:
#%run ./env ----- A decommenter pour lancer le notebook separement
#%run ./python_libraries ----- A decommenter pour lancer le notebook separement

## align_target_schema

Aligne le schema de la table Delta cible sur celui du DataFrame, avant le MERGE.

`handle_table_update` fait un MERGE sur une table existante : une colonne presente
dans le DataFrame mais absente de la cible fait echouer le run. 

A appeler juste avant `handle_table_update`. Destine a rejoindre `delta_function`.

In [0]:
def align_target_schema(df, target_table, defaults=None, verbose=False):
    """
    Ajoute a la table Delta cible les colonnes presentes dans le DataFrame mais
    absentes de la cible, puis backfill les lignes existantes.

    df            : le DataFrame qui va etre merge
    target_table  : nom complet catalog.schema.table
    defaults      : {nom_colonne: litteral SQL}, ex {"deleted": "false"}.
                    Une colonne absente du dict est ajoutee sans backfill.

    Les colonnes presentes en cible mais absentes du DataFrame ne sont pas
    touchees : le MERGE les laisse telles quelles.
    """
    if not spark.catalog.tableExists(target_table):
        # Premier run : handle_table_update creera la table au bon schema.
        if verbose:
            print(f"{target_table} : table inexistante, rien a aligner")
        return

    defaults = {k.lower(): v for k, v in (defaults or {}).items()}

    target_columns = {f.name.lower() for f in spark.table(target_table).schema.fields}
    missing = [f for f in df.schema.fields if f.name.lower() not in target_columns]

    if not missing:
        if verbose:
            print(f"{target_table} : schema deja aligne")
        return

    for field in missing:
        type_sql = field.dataType.simpleString()
        spark.sql(f"ALTER TABLE {target_table} ADD COLUMN `{field.name}` {type_sql}")
        print(f"{target_table} : colonne `{field.name}` ({type_sql}) ajoutee")

        default_value = defaults.get(field.name.lower())
        if default_value is not None:
            spark.sql(
                f"UPDATE {target_table} "
                f"SET `{field.name}` = {default_value} "
                f"WHERE `{field.name}` IS NULL"
            )
            print(f"{target_table} : `{field.name}` backfill a {default_value}")
        else:
            print(
                f"{target_table} : `{field.name}` sans valeur par defaut, "
                f"les lignes existantes restent a NULL"
            )
